# Report Structure

This notebook explores the current ClimateKG knowledge graph by tracing the report hierarchy using the live Wikibase data.

We start with the relationship that connects an item to its parent in the corpus structure: `P3`.

**Query UI**: https://climatekg.tibwiki.io/query/

**SPARQL endpoint**: https://climatekg.tibwiki.io/query/proxy/sparql

This notebook is intentionally query-first: we validate the structure before adding deeper interpretation.

In [1]:
import os
import pandas as pd
from SPARQLWrapper import JSON
from IPython.display import display, Markdown

from wikibase_auth import DEFAULT_SPARQL_ENDPOINT, DEFAULT_WIKIBASE_URL, build_sparql_client

os.environ.setdefault("CLIMATEKG_SPARQL_USERNAME", "ckg")
os.environ.setdefault("CLIMATEKG_SPARQL_PASSWORD", "fairdata")

SPARQL_ENDPOINT = os.getenv("CLIMATEKG_SPARQL_ENDPOINT", DEFAULT_SPARQL_ENDPOINT)
WIKIBASE_URL = os.getenv("CLIMATEKG_WIKIBASE_URL", DEFAULT_WIKIBASE_URL)

print(f"Endpoint: {SPARQL_ENDPOINT}")
print("Libraries imported successfully")

'ckg'

'fairdata'

Endpoint: https://climatekg.tibwiki.io/query/proxy/sparql
Libraries imported successfully


In [2]:
query = '''
PREFIX wd: <https://climatekg.tibwiki.io/entity/>
PREFIX wdt: <https://climatekg.tibwiki.io/prop/direct/>

SELECT ?item ?itemLabel ?parent ?parentLabel ?itemType ?itemTypeLabel
WHERE {
  ?item wdt:P3 ?parent .
  OPTIONAL { ?item wdt:P1 ?itemType . }
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
}
ORDER BY ?parent ?itemLabel
LIMIT 200
'''

sparql = build_sparql_client(SPARQL_ENDPOINT)
sparql.setQuery(query)
sparql.setReturnFormat(JSON)
results = sparql.query().convert()
rows = results.get("results", {}).get("bindings", [])
df = pd.DataFrame([{k: v["value"] for k, v in row.items()} for row in rows])

print(f"Rows returned: {len(df)}")
display(df.head(20))

Rows returned: 3


,item,parent,itemType,itemLabel,parentLabel,itemTypeLabel
0,https://climatekg.tibwiki.io/entity/Q188,https://climatekg.tibwiki.io/entity/Q153,https://climatekg.tibwiki.io/entity/Q6,Accelerating the Transition in the Context of ...,https://climatekg.tibwiki.io/entity/Q153,https://climatekg.tibwiki.io/entity/Q6
1,https://climatekg.tibwiki.io/entity/Q190,https://climatekg.tibwiki.io/entity/Q189,https://climatekg.tibwiki.io/entity/Q6,Summary for Policymakers,https://climatekg.tibwiki.io/entity/Q189,https://climatekg.tibwiki.io/entity/Q6
2,https://climatekg.tibwiki.io/entity/Q192,https://climatekg.tibwiki.io/entity/Q191,https://climatekg.tibwiki.io/entity/Q6,Longer Report,https://climatekg.tibwiki.io/entity/Q191,https://climatekg.tibwiki.io/entity/Q6


## Interpretation

Each row represents an item with a parent item in the report hierarchy. In the current ClimateKG graph this gives us the basic structure for a report tree: the item becomes a child of the parent node via `P3`.

This is the right foundation for the report-structure notebook because it shows how reports, sections, and related objects are connected in context rather than as isolated records.

The next step could be to aggregate these rows by parent and count child items, or to add a direct graph visualisation of the hierarchy.